In [1]:
import pyspark
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, length, trim

# Initialise Spark
spark = SparkSession.builder.appName("Zero_Trust_EDA").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 1. Added the missing LMS CSV
df_attr = spark.read.csv("data/features_attributes.csv", header=True, inferSchema=True)
df_fin = spark.read.csv("data/features_financials.csv", header=True, inferSchema=True)
df_click = spark.read.csv("data/feature_clickstream.csv", header=True, inferSchema=True)
df_lms = spark.read.csv("data/lms_loan_daily.csv", header=True, inferSchema=True)

print("=== Check customers ===")

# Check Rows vs Unique IDs (Added Attributes and LMS for complete visibility)
print(f"Attributes:  {df_attr.count()} Rows | {df_attr.select('Customer_ID').distinct().count()} Unique IDs")
print(f"Financials:  {df_fin.count()} Rows | {df_fin.select('Customer_ID').distinct().count()} Unique IDs")
print(f"Clickstream: {df_click.count()} Rows | {df_click.select('Customer_ID').distinct().count()} Unique IDs")
print(f"LMS Ledger:  {df_lms.count()} Rows | {df_lms.select('Customer_ID').distinct().count()} Unique IDs")

# ID Format Check (Must be exactly 10 characters)
print("\nChecking ID Integrity (Should be 0):")
print("Malformed IDs in Attributes:", df_attr.filter(length(trim(col("Customer_ID"))) != 10).count())
print("Malformed IDs in Financials:", df_fin.filter(length(trim(col("Customer_ID"))) != 10).count())
print("Malformed IDs in Clickstream:", df_click.filter(length(trim(col("Customer_ID"))) != 10).count())
print("Malformed IDs in LMS Ledger:", df_lms.filter(length(trim(col("Customer_ID"))) != 10).count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/16 12:43:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

=== Check customers ===


Attributes:  12500 Rows | 12500 Unique IDs
Financials:  12500 Rows | 12500 Unique IDs
Clickstream: 215376 Rows | 8974 Unique IDs
LMS Ledger:  137500 Rows | 12500 Unique IDs

Checking ID Integrity (Should be 0):
Malformed IDs in Attributes: 756
Malformed IDs in Financials: 756
Malformed IDs in Clickstream: 13152
Malformed IDs in LMS Ledger: 8316


In [2]:
print("Check for dirty data")

fake_null_list = ["NM", "N/A", "NA", "null", "NULL", "-", "None", "", "Not Specified"]
# Regex to catch weird symbols: Allows letters, numbers, spaces, dots, and dashes. Flags everything else.
garbage_regex = r"[^a-zA-Z0-9 .\-]" 

def sweep_dataframe(df, df_name):
    print(f"\n--- Sweeping {df_name} ---")
    string_cols = [c for c, t in df.dtypes if t == 'string']
    
    for c in string_cols:
        # 1. Catch Fake Nulls
        fake_nulls = df.filter(col(c).isin(fake_null_list) | col(c).rlike(r"^\s+$")).count()
        # 2. Catch Underscores (Properly escaped)
        underscores = df.filter(col(c).rlike(r"\_")).count()
        # 3. Catch Weird Symbols (like !@9#%8)
        weird_symbols = df.filter(col(c).rlike(garbage_regex) & ~col(c).rlike(r"\_")).count()
        
        if fake_nulls > 0 or underscores > 0 or weird_symbols > 0:
            print(f"[{c}] -> Fake Nulls: {fake_nulls} | Underscores: {underscores} | Garbage Symbols: {weird_symbols}")

sweep_dataframe(df_attr, "Attributes")
sweep_dataframe(df_fin, "Financials")
# 1. Added the sweep for the LMS ledger
sweep_dataframe(df_lms, "LMS Ledger")

Check for dirty data

--- Sweeping Attributes ---
[Customer_ID] -> Fake Nulls: 0 | Underscores: 12500 | Garbage Symbols: 0
[Name] -> Fake Nulls: 0 | Underscores: 0 | Garbage Symbols: 119
[Age] -> Fake Nulls: 0 | Underscores: 637 | Garbage Symbols: 0
[SSN] -> Fake Nulls: 0 | Underscores: 0 | Garbage Symbols: 703
[Occupation] -> Fake Nulls: 0 | Underscores: 1660 | Garbage Symbols: 0

--- Sweeping Financials ---
[Customer_ID] -> Fake Nulls: 0 | Underscores: 12500 | Garbage Symbols: 0
[Annual_Income] -> Fake Nulls: 0 | Underscores: 859 | Garbage Symbols: 0
[Num_of_Loan] -> Fake Nulls: 0 | Underscores: 623 | Garbage Symbols: 0
[Type_of_Loan] -> Fake Nulls: 176 | Underscores: 0 | Garbage Symbols: 9683
[Num_of_Delayed_Payment] -> Fake Nulls: 0 | Underscores: 374 | Garbage Symbols: 0
[Changed_Credit_Limit] -> Fake Nulls: 0 | Underscores: 254 | Garbage Symbols: 0
[Credit_Mix] -> Fake Nulls: 0 | Underscores: 2611 | Garbage Symbols: 0
[Outstanding_Debt] -> Fake Nulls: 0 | Underscores: 139 | Garba

In [9]:
print("Check categorical values")

# 1. Binary/Boolean Check (Should only be Yes/No/NM/etc)
print("Values in Payment_of_Min_Amount (Expected: Yes/No):")
df_fin.groupBy("Payment_of_Min_Amount").count().orderBy("count").show()

# 2. Category Check for Dirty Categories
print("\nValues in Credit_Mix:")
df_fin.groupBy("Credit_Mix").count().orderBy("count").show()

print("\nValues in Payment_Behaviour:")
df_fin.groupBy("Payment_Behaviour").count().orderBy("count").show()

# 3. Concatenation Check
print("\nTop 5 Longest Strings in Type_of_Loan (Checking for comma-separated lists):")
df_fin.withColumn("len", length(col("Type_of_Loan")))\
      .orderBy(F.desc("len"))\
      .select("Type_of_Loan", "len").show(5, truncate=False)

Check categorical values
Values in Payment_of_Min_Amount (Expected: Yes/No):
+---------------------+-----+
|Payment_of_Min_Amount|count|
+---------------------+-----+
|                   NM| 1438|
|                   No| 4491|
|                  Yes| 6571|
+---------------------+-----+


Values in Credit_Mix:
+----------+-----+
|Credit_Mix|count|
+----------+-----+
|       Bad| 2360|
|         _| 2611|
|      Good| 3032|
|  Standard| 4497|
+----------+-----+


Values in Payment_Behaviour:
+--------------------+-----+
|   Payment_Behaviour|count|
+--------------------+-----+
|              !@9#%8|  998|
|Low_spent_Large_v...| 1300|
|High_spent_Small_...| 1389|
|High_spent_Large_...| 1683|
|Low_spent_Medium_...| 1686|
|High_spent_Medium...| 2242|
|Low_spent_Small_v...| 3202|
+--------------------+-----+


Top 5 Longest Strings in Type_of_Loan (Checking for comma-separated lists):
+------------------------------------------------------------------------------------------------------------

In [4]:
from pyspark.sql.types import DoubleType, IntegerType # Changed from FloatType to prevent rounding errors

print("=== Check numerical values and outliers ===")

# Temporarily strip non-numeric characters from Financials so we can calculate Min/Max
df_fin_math = df_fin
numeric_targets = ["Annual_Income", "Monthly_Inhand_Salary", "Num_Bank_Accounts", 
                   "Num_Credit_Card", "Interest_Rate", "Delay_from_due_date", 
                   "Credit_Utilization_Ratio", "Outstanding_Debt"]

for c in numeric_targets:
    # Scrub out anything that isn't a number, negative sign, or decimal, then cast to DoubleType
    df_fin_math = df_fin_math.withColumn(c, F.regexp_replace(col(c), "[^0-9.-]", "").cast(DoubleType()))

# Now calculate the real boundaries
df_fin_math.select(numeric_targets).describe().show(vertical=True)

# Do the same check for Age in Attributes (Cast to Integer)
df_attr_math = df_attr.withColumn("Age", F.regexp_replace(col("Age"), "[^0-9]", "").cast(IntegerType()))
df_attr_math.select("Age").describe().show()

# 1. ADDED: Check boundaries for the new LMS Ledger file
print("\n=== Check numerical boundaries for LMS Ledger ===")
df_lms.select("loan_amt", "due_amt", "paid_amt", "overdue_amt", "balance").describe().show()

=== Check numerical values and outliers ===
-RECORD 0--------------------------------------
 summary                  | count              
 Annual_Income            | 12500              
 Monthly_Inhand_Salary    | 12500              
 Num_Bank_Accounts        | 12500              
 Num_Credit_Card          | 12500              
 Interest_Rate            | 12500              
 Delay_from_due_date      | 12500              
 Credit_Utilization_Ratio | 12500              
 Outstanding_Debt         | 12500              
-RECORD 1--------------------------------------
 summary                  | mean               
 Annual_Income            | 161620.55254240037 
 Monthly_Inhand_Salary    | 4188.5923027131585 
 Num_Bank_Accounts        | 16.93992           
 Num_Credit_Card          | 23.17272           
 Interest_Rate            | 73.21336           
 Delay_from_due_date      | 21.06088           
 Credit_Utilization_Ratio | 32.34926457800977  
 Outstanding_Debt         | 1426.22037600000

In [11]:
import os
import shutil
import pyspark
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Make sure we have the function imported (assuming it's in the same notebook or you can paste it above this)
from utils.data_processing_feature_silver import process_silver_features

spark = SparkSession.builder.appName("Silver_Unit_Test").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Testing data preprocessing ETL")

# Test datamart
test_date = "2099-01-01"
bronze_dir = "test_datamart/bronze/"
silver_dir = "test_datamart/silver/"
os.makedirs(bronze_dir, exist_ok=True)
os.makedirs(silver_dir, exist_ok=True)

# Mock Attributes (Includes a valid ID, a 9-char malformed ID, poisoned age, and a 4000-yr-old)
attr_data = [
    ("CUS_0x1000", "40_"),   # Valid ID, poisoned age
    ("CUS_0x100", "25"),     # Malformed ID (9 chars)
    ("CUS_0x2000", "8678")   # Valid ID, impossible age
]
df_attr_mock = spark.createDataFrame(attr_data, ["Customer_ID", "Age"])

# Mock Financials (Includes poisoned income, negative bank accounts, 1000 bank accounts, garbage categories, and concatenations)
fin_data = [
    ("CUS_0x1000", "52312.68_", -1, "NULL", "_", "!@9#%8", "NM"), 
    ("CUS_0x100", "50000", 2, "Auto Loan", "Good", "Standard", "Yes"), 
    ("CUS_0x2000", "10000", 1756, "Auto Loan, Payday Loan", "Good", "Standard", "No")
]
fin_cols = ["Customer_ID", "Annual_Income", "Num_Bank_Accounts", "Type_of_Loan", "Credit_Mix", "Payment_Behaviour", "Payment_of_Min_Amount"]
df_fin_mock = spark.createDataFrame(fin_data, fin_cols)

# Mock Clickstream (Includes duplicate clicks for one user, a malformed user, and a valid "Ghost" user with no profile)
click_data = [
    ("CUS_0x1000", 5),
    ("CUS_0x1000", 10),      # Duplicate row
    ("CUS_0x100", 2),        # Malformed ID
    ("CUS_0x9999", 8)        # Ghost ID (exists here, but not in attributes)
]
df_click_mock = spark.createDataFrame(click_data, ["Customer_ID", "Clicks"])

# Save mocks to temporary bronze folder
df_attr_mock.write.mode("overwrite").csv(bronze_dir + "bronze_attributes_2099_01_01.csv", header=True)
df_fin_mock.write.mode("overwrite").csv(bronze_dir + "bronze_financials_2099_01_01.csv", header=True)
df_click_mock.write.mode("overwrite").csv(bronze_dir + "bronze_clickstream_2099_01_01.csv", header=True)

# Run silver layer
print("Running Silver Layer Processing...")
df_result = process_silver_features(test_date, bronze_dir, silver_dir, spark)

# Validation check
print("Running Strict Validations...\n")
results = df_result.collect()

# Helper function to get a specific customer's row
def get_cust(customer_id):
    for r in results:
        if r["Customer_ID"] == customer_id:
            return r
    return None

try:
    # Test A: Did the Malformed ID get dropped across all tables?
    assert get_cust("CUS_0x100") is None, "FAIL: Malformed ID 'CUS_0x100' leaked through."
    
    # Test B: Did the Cartesian Explosion get stopped? (Total rows should only be 3 unique valid IDs)
    assert df_result.count() == 3, f"FAIL: Expected 3 rows, found {df_result.count()}. Deduplication failed."
    
    cust_1000 = get_cust("CUS_0x1000")
    cust_2000 = get_cust("CUS_0x2000")
    cust_9999 = get_cust("CUS_0x9999")

    # Test C: Numeric Scrubbing
    assert cust_1000["Age"] == 40, f"FAIL: Age underscore not scrubbed. Got {cust_1000['Age']}"
    assert cust_1000["Annual_Income"] == 52312.68, "FAIL: Income underscore not scrubbed."

    # Test D: Outlier Capping (Reality Check)
    assert cust_2000["Age"] is None, "FAIL: 8000-year-old age was not set to None/null."
    assert cust_1000["Num_Bank_Accounts"] == 0, "FAIL: Negative bank accounts not capped to 0."
    assert cust_2000["Num_Bank_Accounts"] is None, "FAIL: 1756 bank accounts not capped to None/null."

    # Test E: Categorical Fixing
    assert cust_1000["Type_of_Loan"] == "No Loan", "FAIL: 'NULL' string not mapped to 'No Loan'."
    assert cust_2000["Type_of_Loan"] == "Multiple Loans", "FAIL: Comma-separated loans not mapped to 'Multiple Loans'."
    assert cust_1000["Credit_Mix"] == "Unknown", "FAIL: '_' not mapped to 'Unknown'."
    assert cust_1000["Payment_Behaviour"] == "Unknown", "FAIL: '!@9#%8' not mapped to 'Unknown'."
    assert cust_1000["Payment_of_Min_Amount"] == "Unknown", "FAIL: 'NM' not mapped to 'Unknown'."

    # Test F: Ghost Customer (Outer Join Null Handling)
    assert cust_9999["Payment_Behaviour"] == "Unknown", "FAIL: Ghost customer string cols were not filled with 'Unknown'."
    assert cust_9999["Age"] is None, "FAIL: Ghost customer numeric cols should be null, but aren't."

    print("Test passed.")

except AssertionError as e:
    print("failed:")
    print(e)

# Cleanup the test directory
finally:
    if os.path.exists("test_datamart"):
        shutil.rmtree("test_datamart")

Testing data preprocessing ETL


Running Silver Layer Processing...
saved to: test_datamart/silver/silver_features_2099_01_01.parquet
Running Strict Validations...

Test passed.
